# Обучение U-Net (ResNet34) для сегментации озёр — Google Colab

Ноутбук запускает **только обучение** уже готового пайплайна из репозитория
`sentinel2-lake-segmentation`: загрузка сцен Sentinel-2 → препроцессинг → обучение →
оценка → экспорт ONNX → сбор артефактов.

**Порядок действий**

1. `Среда выполнения → Сменить тип среды выполнения → T4 GPU` (обучение на CPU возможно, но
   ~20–30 раз медленнее).
2. Загрузите архив репозитория `sentinel2-lake-segmentation.zip` в файлы Colab (иконка папки
   слева) **или** укажите `REPO_URL` вашего GitHub-репозитория в ячейке «1. Код».
3. `Среда выполнения → Выполнить все` (`Ctrl+F9`) и дождитесь конца. Полный прогон на T4:
   ~5 мин на данные + ~10–15 мин на обучение + ~2 мин на экспорт.
4. Последняя ячейка скачает `lake_segmentation_artifacts.zip` — **это и есть результат**,
   который нужно вернуть агенту (веса, ONNX, датасет, графики, MLflow-база).

Ноутбук идемпотентен: повторный запуск переиспользует скачанные данные и переобучит модель.

## 0. Проверка среды и GPU

In [ ]:
import subprocess

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
        capture_output=True,
        text=True,
    ).stdout
)

import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

## 1. Код проекта

Выберите **один** из двух вариантов:

* `REPO_URL` — ссылка на GitHub-репозиторий (рекомендуется, если вы уже запушили код);
* `REPO_ARCHIVE` — путь к загруженному zip-архиву (`/content/sentinel2-lake-segmentation.zip`).

Если оба пусты, ячейка попросит загрузить архив через диалог.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

REPO_URL = ""      # например "https://github.com/<account>/sentinel2-lake-segmentation.git"
REPO_ARCHIVE = ""  # например "/content/sentinel2-lake-segmentation.zip"
REPO_DIR = Path("/content/sentinel2-lake-segmentation")


def extract_archive(archive_path: Path) -> None:
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.namelist()
        prefix = members[0].split("/")[0] if members else ""
        archive.extractall(REPO_DIR.parent)
        extracted = REPO_DIR.parent / prefix
        if prefix and extracted.is_dir() and not REPO_DIR.exists():
            extracted.rename(REPO_DIR)


if REPO_URL:
    if (REPO_DIR / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
elif REPO_ARCHIVE and Path(REPO_ARCHIVE).is_file():
    extract_archive(Path(REPO_ARCHIVE))
else:
    from google.colab import files  # noqa: E402

    print("Загрузите архив репозитория (sentinel2-lake-segmentation.zip):")
    uploaded = files.upload()
    archive_name = next(iter(uploaded))
    extract_archive(Path(archive_name))
    if archive_name.startswith("/content") is False:
        Path(archive_name).unlink(missing_ok=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print("рабочая директория:", Path.cwd())
print("содержимое:", sorted(path.name for path in Path.cwd().iterdir()))

## 2. Установка зависимостей

Ставим пакет в editable-режиме: версии зависимостей описаны в `pyproject.toml`
(`uv.lock` фиксирует их для локальной установки через `uv sync`). Colab уже содержит
подходящий `torch`, поэтому заново он не скачивается.

In [ ]:
%pip install -q -e .

import lake_segmentation
import lightning

print("lake_segmentation", lake_segmentation.__version__)
print("lightning", lightning.__version__)

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "commands.py", "--help"], capture_output=True, text=True
)
print(result.stdout[-1500:] or result.stderr[-1500:])

## 3. MLflow tracking server

Агент ожидает сервер по адресу `127.0.0.1:8080` (см. `configs/mlflow/default.yaml`).
Поднимаем его локально в Colab; артефакты складываются в `mlruns/`, база — в `mlflow.db`,
поэтому всё это попадёт в итоговый архив.

In [ ]:
import socket
import subprocess
import time
from pathlib import Path

MLFLOW_LOG = Path("/content/mlflow.log")
mlflow_process = subprocess.Popen(
    [
        "mlflow", "server",
        "--host", "127.0.0.1",
        "--port", "8080",
        "--backend-store-uri", f"sqlite:///{Path.cwd() / 'mlflow.db'}",
        "--default-artifact-root", str(Path.cwd() / "mlruns"),
    ],
    stdout=MLFLOW_LOG.open("w"),
    stderr=subprocess.STDOUT,
)

for attempt in range(120):
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", 8080)) == 0:
            print(f"MLflow запущен на http://127.0.0.1:8080 (pid {mlflow_process.pid})")
            break
    time.sleep(1)
else:
    print("MLflow не поднялся, обучение продолжится с CSV-логгером")
    print(MLFLOW_LOG.read_text()[-2000:])

## 4. Данные: поиск сцен и препроцессинг

`download` находит лучшую сцену Sentinel-2 для каждого озера и проверяет доступность каналов,
`preprocess` собирает фрагменты 256×256, маски WorldCover, разбиение по озёрам и статистики
нормализации. Данные не коммитятся в git — они управляются DVC.

Ожидаемое время: 3–8 минут (скачивается ~400 МБ фрагментов сцен по HTTP).

In [ ]:
import subprocess
import sys

def run(command: list[str]) -> None:
    print(f"\n$ {' '.join(command)}\n")
    subprocess.run(command, check=True)

run([sys.executable, "commands.py", "download"])
run([sys.executable, "commands.py", "preprocess"])

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path("data/processed/dataset_summary.json").read_text())
print("патчей всего:", summary["num_patches"])
for split in ("train", "val", "test"):
    info = summary["splits"].get(split, {})
    print(f"  {split:5s}: {info.get('patches', 0):4d} патчей, озёра {info.get('lakes', [])}")
if summary.get("split_assignment"):
    print("принадлежность озёр:", summary["split_assignment"])
print(
    "нормализация:",
    {key: [round(value, 4) for value in values]
     for key, values in summary["normalization"].items()},
)


## 5. Обучение

Гиперпараметры — в `configs/train/default.yaml`, здесь переопределяются под T4
(batch 32, mixed precision, 30 эпох). Модель: U-Net + ResNet34 (ImageNet), лосс BCE + Dice,
AdamW с cosine-расписанием, early stopping по `val/iou`, порог бинаризации подбирается на
валидации в конце обучения.

In [ ]:
TRAIN_OVERRIDES = ",".join([
    "train.max_epochs=30",
    "train.batch_size=32",
    "train.num_workers=2",
    "train.precision=16-mixed",
    "train.accelerator=gpu",
    "train.devices=1",
    "mlflow.tracking_uri=http://127.0.0.1:8080",
])

run([
    sys.executable, "commands.py", "train",
    "--overrides", TRAIN_OVERRIDES,
])

## 6. Оценка на тесте и NDWI-бейзлайн

In [ ]:
run([sys.executable, "commands.py", "evaluate"])
run([sys.executable, "commands.py", "baseline"])

## 7. Подготовка к продакшену: ONNX (и команда TensorRT)

ONNX экспортируется вместе с сигмоидой и проверяется против PyTorch через `onnxruntime`.
TensorRT на Colab недоступен (`trtexec` не установлен), поэтому показываем команду сборки
(`--dry-run`); сам движок собирается на машине с GPU NVIDIA и TensorRT.

In [ ]:
run([sys.executable, "commands.py", "export-onnx"])
run([sys.executable, "commands.py", "export-tensorrt", "--dry-run"])

## 8. Результаты

Метрики на тестовой выборке, графики обучения и предсказания.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

bundle = json.loads(Path("weights/bundle.json").read_text())
test_metrics = bundle["metrics"]["test"]
print(f"порог бинаризации : {bundle['postprocessing']['threshold']:.3f}")
print(f"IoU               : {test_metrics['iou']:.4f}")
print(f"Dice              : {test_metrics['dice']:.4f}")
print(f"Precision         : {test_metrics['precision']:.4f}")
print(f"Recall            : {test_metrics['recall']:.4f}")
print(f"Relative area err : {test_metrics['area_error_percent']:.2f} %")
print("\nпо озёрам:")
for lake, metrics in bundle["metrics"]["per_lake_test"].items():
    print(
        f"  {lake:14s} IoU={metrics['iou']:.3f} Dice={metrics['dice']:.3f} "
        f"area_err={metrics['area_error_percent']:.1f}%"
    )

In [ ]:
for name in ("losses.png", "metrics.png", "area_error.png", "learning_rate.png",
             "val_predictions.png", "dataset_preview.png"):
    path = Path("plots") / name
    if path.is_file():
        display(Image(filename=str(path), width=900))

In [ ]:
baseline_path = Path("reports/ndwi_baseline.json")
if baseline_path.is_file():
    print(json.dumps(json.loads(baseline_path.read_text())["test"], indent=2))

## 9. Сбор артефактов

В архив попадают: веса и ONNX, `bundle.json`, препроцессированный датасет, графики, отчёты,
MLflow-база и артефакты MLflow. Этого достаточно, чтобы продолжить работу локально:
проверить инференс, залить данные в DVC и заполнить README результатами.

In [ ]:
import subprocess
import zipfile
from pathlib import Path

ARCHIVE_PATH = Path("/content/lake_segmentation_artifacts.zip")
INCLUDE = [
    "weights",
    "data/processed",
    "data/raw",
    "plots",
    "reports",
    "predictions",
    "mlruns",
    "logs",
    "mlflow.db",
    "mlflow-artifacts",
]

with zipfile.ZipFile(ARCHIVE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for entry in INCLUDE:
        path = Path(entry)
        if path.is_file():
            archive.write(path, path.name)
        elif path.is_dir():
            for file in path.rglob("*"):
                if file.is_file():
                    archive.write(file, str(file))

size_mb = ARCHIVE_PATH.stat().st_size / 1e6
print(f"архив: {ARCHIVE_PATH} ({size_mb:.1f} МБ)")
print(subprocess.run(["du", "-sh", *INCLUDE], capture_output=True, text=True).stdout)

In [ ]:
# Скачать архив на компьютер (в браузере появится сохранение файла).
from google.colab import files

files.download(str(ARCHIVE_PATH))

In [ ]:
# Необязательно: сохранить архив на Google Drive, если скачивание обрывается.
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    target = Path("/content/drive/MyDrive/lake_segmentation_artifacts.zip")
    target.write_bytes(ARCHIVE_PATH.read_bytes())
    print("сохранено в", target)

## Что дальше

1. Скачанный `lake_segmentation_artifacts.zip` положите в папку проекта
   `~/Develop/Study/mlops/` (рядом с `sentinel2-lake-segmentation/`).
2. Сообщите агенту, что архив готов и обучение завершено.
3. Агент распакует архив, зарегистрирует данные и веса в DVC, проверит инференс на паре дат,
   сравнит с бейзлайном, заполнит раздел «Результаты» в README и закоммитит изменения.

Если обучение упало, скопируйте текст ошибки из ячейки — по нему видно, на каком этапе
проблема (данные, обучение, экспорт).